# N040 · 堆排序与非比较排序

**目标：** 依据键类型和值域选择排序机制。

**先修：** N037, N039, N011。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 堆结构导览；计数/桶/基数；稳定子过程；比较下界适用条件。

## 从问题到算法

堆排序只要求父节点不小于子节点，不要求整段有序。建好最大堆后，把堆顶交换到末尾，再修复缩小的堆。非比较排序利用额外值域结构：计数排序依赖值域R；基数排序按低位到高位做稳定分桶。禁止对巨大稀疏值域无条件分配数组。

## 最小实现

**本章接口：** `heap_sort(a)`、`counting_sort(a)`、`radix_sort_nonnegative(a)`

In [1]:
def heap_sort(a):
    def sift(root,size):
        while 2*root+1<size:
            child=2*root+1
            if child+1<size and a[child+1]>a[child]: child+=1
            if a[root]>=a[child]: break
            a[root],a[child]=a[child],a[root]; root=child
    for i in range(len(a)//2-1,-1,-1): sift(i,len(a))
    for end in range(len(a)-1,0,-1):
        a[0],a[end]=a[end],a[0]; sift(0,end)
    return a

def counting_sort(a,key=lambda x:x,max_range=1000000):
    if not a: return []
    low=min(map(key,a)); high=max(map(key,a)); width=high-low+1
    if width>max_range: raise ValueError('value range too large for counting sort')
    counts=[0]*width
    for x in a: counts[key(x)-low]+=1
    for i in range(1,width): counts[i]+=counts[i-1]
    out=[None]*len(a)
    for x in reversed(a):
        i=key(x)-low; counts[i]-=1; out[counts[i]]=x
    return out

def radix_sort_nonnegative(a):
    if any(x<0 for x in a): raise ValueError('nonnegative integers required')
    out=list(a); place=1; maximum=max(out,default=0)
    while place<=maximum:
        buckets=[[] for _ in range(10)]
        for x in out: buckets[x//place%10].append(x)
        out=[x for bucket in buckets for x in bucket]; place*=10
    return out

## 正确性、前提与复杂度

堆顶为剩余最大值，逐个放入已排序后缀保持归纳关系。基数分桶稳定，使新位相同的元素保持低位已排好的次序，逐位归纳得到整体有序。

**代价：** 堆 O(n log n)时间/O(1)辅助空间；计数 O(n+R)时间空间；d位十进制基数 O(d(n+10))时间/O(n+10)空间。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[170,45,75,90,802,24,2,66]; rows=[]; out=a[:]
for place in [1,10,100]:
    buckets=[[] for _ in range(10)]
    for x in out: buckets[x//place%10].append(x)
    out=[x for b in buckets for x in b]; rows.append((place,out[:]))
show_table(['当前位权','稳定分桶后'],rows)

当前位权,稳定分桶后
1,"[170, 90, 802, 2, 24, 45, 75, 66]"
10,"[802, 2, 24, 45, 66, 170, 75, 90]"
100,"[2, 24, 45, 66, 75, 90, 170, 802]"


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
from random import Random
rng=Random(40)
for n in range(50):
    a=[rng.randrange(1000) for _ in range(n)]
    assert heap_sort(a[:])==counting_sort(a)==radix_sort_nonnegative(a)==sorted(a)
a=[(2,'a'),(1,'b'),(2,'c')]
assert counting_sort(a,key=lambda x:x[0])==[(1,'b'),(2,'a'),(2,'c')]
for fn,values in [(radix_sort_nonnegative,[1,-1]),(counting_sort,[0,10**9])]:
    try: fn(values)
    except ValueError: pass
    else: raise AssertionError('unsupported domain must fail explicitly')
print('N040: 所有本章断言通过')

N040: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 支持有负数的计数排序。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 解释桶排序平均线性需要什么假设。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 912. Sort an Array（comparison）
- 164. Maximum Gap（extension）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。